<h2>5.1 Ausgangspunkt der Analyse</h2>

<p>Die Theorie hat bereits die Bausteine festgelegt, die das spätere Modell benötigt. Dazu gehören die Verarbeitung von Zeitreihen, Renditeberechnungen, Volatilität, Drawdown, Sharpe Ratio, Korrelation, Portfoliorisiko, Rebalancing, Buy-and-Hold und eine Trendfolgestrategie. Die Methodik verlangt ausserdem einen vergleichbaren Zeitabschnitt, dasselbe Startkapital, die Reinvestition der Dividenden und eine einheitliche Datenfrequenz. Beim 60/40-Portfolio ist ein jährliches Rebalancing vorgesehen.</p>

<p>Damit besteht meine Aufgabe in der Analyse nicht darin, eine völlig neue Finanztheorie zu entwickeln. Ich muss die bereits beschriebenen Bestandteile so zusammenführen, dass sie mit realen historischen Daten reproduzierbar ausgeführt werden können.</p>

<p>Dabei unterscheide ich zwei Ebenen:<br>
1. <strong>Das allgemeine System:</strong> Es soll verschiedene geeignete Datensätze und Parameter verarbeiten können.<br>
2. <strong>Die Untersuchung der Maturarbeit:</strong> Sie verwendet später eine feste und begründete Auswahl von Daten und Parametern.</p>

<p>Diese Trennung ist wichtig, weil die spätere Benutzeroberfläche dieselbe Engine mit anderen Eingaben verwenden können soll. Für die wissenschaftliche Auswertung werden die Parameter dagegen vor dem Hauptlauf festgelegt und nicht nachträglich so verändert, dass ein günstigeres Resultat entsteht. Das entspricht auch der im Theorieteil behandelten Gefahr von Data Snooping und Overfitting [@hilpischPythonFinanceMastering2019, S. 697; @kommerSouveraenInvestierenMit2025, S. 607–608].</p>

<h2>5.2 Reihenfolge der technischen Entwicklung</h2>
<p>Aus den Anforderungen ergibt sich für mich folgende Reihenfolge:</p>

$$
\begin{array}{c}
\boxed{\text{\Large 1. Internes Datenformat festlegen}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{\Large 2. Datenquellen auf dieses Format abbilden}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{\Large 3. Import, Normalisierung und Validierung definieren}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{\Large 4. Veränderbare Einstellungen des Systems festlegen}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{\Large 5. Einheitlichen Ergebnisvertrag festlegen}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{\Large 6. Leeres Grundgerüst der Engine definieren}}
\\[0.4cm]
\downarrow
\\[0.4cm]
\boxed{\text{\Large 7. Vorhandene Funktionen integrieren}}
\end{array}
$$

<p>Die Reihenfolge ist bewusst so gewählt. Würde ich zuerst die Simulationsfunktion programmieren und erst danach entscheiden, welche Eingaben und Ausgaben sie besitzen soll, würden Datenformat, Strategien und Benutzeroberfläche unnötig voneinander abhängig.</p>
<p>Die konkrete Reihenfolge ist meine eigene Designentscheidung. Die Quellen liefern die Grundlagen dafür, welche Datenstrukturen und Finanzinformationen benötigt werden; sie schreiben nicht diese exakte Softwarearchitektur vor.</p>

<h2>5.3 Entwicklung eines einheitlichen Datenformats</h2>
<h3>5.3.1 Warum überhaupt ein eigenes internes Format?</h3>
<p>Die später verwendeten Daten können aus unterschiedlichen Quellen stammen. Yahoo Finance liefert Marktzeitreihen anders als FRED eine Zinsreihe oder eine Makrodatenbank jährliche BIP-Werte. Würde jede Strategie direkt mit dem Format einer bestimmten Quelle arbeiten, müsste sich auch die Strategie ändern, sobald der Anbieter gewechselt wird.</p>
<p>Deshalb erhält das Modell einen <strong>anbieterunabhängigen Datenvertrag</strong>. Ein Importmodul ist dafür verantwortlich, Quelldaten in diesen Vertrag zu übersetzen. Die eigentliche Strategie sieht danach nur noch standardisierte Daten.</p>
<p>Wes McKinney beschreibt Datenanalyse als Abfolge von Einlesen, Bereinigen, Kombinieren, Normalisieren, Umformen und anschliessender Berechnung. Er behandelt dabei sowohl tabellarische Daten als auch Zeitreihen und zeigt, dass pandas gerade für solche strukturierten Daten entwickelt wurde [@mckinneyPythonDataAnalysis2022]. Diese Grundidee passt zu meiner Untersuchung: Die Quelle darf unterschiedlich sein, die Berechnung soll danach aber auf einer einheitlichen Struktur stattfinden.</p>
<p>Die Entscheidung, dafür einen eigenen Datenvertrag zu definieren, ist meine technische Schlussfolgerung aus diesem Bedarf.</p>

<h3>5.3.2 Warum CSV als Austauschformat?</h3>
<p>Für den Austausch zwischen Import, Kontrolle und späterer Benutzeroberfläche verwende ich in der ersten Version CSV-Dateien. McKinney behandelt in Kapitel 6 das Einlesen tabellarischer Textdaten und verwendet `pandas.read_csv()` als eines der zentralen Werkzeuge für solche Daten [@mckinneyPythonDataAnalysis2022].</p>
<p>CSV ist für mein Projekt nicht deshalb ausgewählt, weil es allen anderen Formaten technisch überlegen wäre. Die Gründe sind praktischer:</p>

- Die Datei kann direkt mit einem Texteditor, Tabellenprogramm oder Python geöffnet werden.
- Beim Entwickeln kann ich leicht kontrollieren, ob ein Adapter tatsächlich die erwarteten Spalten erzeugt.
- Eine spätere Weboberfläche kann eine CSV-Datei einfach als Benutzereingabe akzeptieren.
- pandas kann das Format ohne zusätzliche Spezialbibliothek einlesen.

<p>CSV besitzt gleichzeitig Grenzen. Datentypen und die Bedeutung einer Spalte werden nicht ausreichend durch die Datei selbst erklärt. Darum genügt eine CSV allein nicht: Das System benötigt zusätzlich einen klar definierten Datenvertrag und Metadaten.</p>
<p>Für die interne Berechnung bleibt die Datei nicht als Textstruktur bestehen. Nach dem Einlesen wird sie in pandas-Objekte überführt.</p>


<h3>5.3.3 Warum ein Long-Format?</h3>
<p>Für die Marktzeitreihen verwende ich als Austauschformat eine lange Tabelle. Eine Beobachtung besteht aus einem Datum, einer Anlagenkennung und einem Wert.</p>
<p>McKinney beschreibt im Abschnitt "Pivoting 'Long' to 'Wide' Format" ausdrücklich, dass mehrere Zeitreihen häufig in einem sogenannten <i>long</i> beziehungsweise <i>stacked format</i> gespeichert werden. In diesem Format stellt jede einzelne Beobachtung eine eigene Zeile dar. Er zeigt danach mit `DataFrame.pivot()`, wie dieselben Daten für Berechnungen in eine breite Tabelle mit einer Spalte pro Zeitreihe umgeformt werden können [@mckinneyPythonDataAnalysis2022].</p>
<p>Diese Beschreibung passt direkt zu meinem Problem. Für ein allgemeines Eingabeformat ist Long praktisch, weil eine neue Anlage zusätzliche Zeilen erzeugt, ohne dass sich die Grundstruktur der Tabelle ändert. Für bestimmte Berechnungen ist dagegen eine breite Darstellung komfortabler. Deshalb lege ich nicht fest, dass jede Stufe des Systems Long sein muss:</p>

$$
\begin{array}{c}
\text{\large \textbf{Austausch / Speicherung:}}
\\[4pt]
\boxed{\texttt{date \;|\; asset\_id \;|\; performance\_value}}
\\[10pt]
\downarrow
\\[-2pt]
\text{\texttt{pandas pivot()}}
\\[10pt]
\downarrow
\\[-2pt]
\text{\large \textbf{Interne Berechnung:}}
\\[4pt]
\boxed{\texttt{date \;|\; ASSET\_A \;|\; ASSET\_B \;|\; ASSET\_C \;|\; \ldots}}
\end{array}
$$

<p>McKinney liefert die nachvollziehbare Grundlage dafür, dass Long ein gebräuchliches Format für mehrere Zeitreihen ist und sich mit pandas eindeutig in Wide umformen lässt. Meine Entscheidung ist, Long deshalb als äussere Schnittstelle und Wide dort zu verwenden, wo die Berechnung davon profitiert.</p>

<h3>5.3.4 Marktzeitreihen: minimaler Datenvertrag</h3>